In [ ]:
#!pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
  Using cached google_auth-2.37.0-py2.py3-none-any.whl (209 kB)
  Using cached google_cloud_aiplatform-1.76.0-py2.py3-none-any.whl (6.9 MB)
  Using cached google_cloud_storage-2.19.0-py2.py3-none-any.whl (131 kB)
  Using cached numpy-2.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
  Using cached pymupdf-1.25.1-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (20.0 MB)
  Using cached pillow-11.1.0-cp310-cp310-manylinux_2_28_x86_64.whl (4.5 MB)


In [2]:
# Import Libraries
import google.auth
from google.cloud import storage, aiplatform
import os
import tensorflow as tf
import tensorflow_hub as hub
import json
import datetime
import dotenv
import logging

from vector_search_image_query import create_gcs_bucket, \
    extract_images_from_pdfs_in_gcs, \
    list_gcs_image_paths, \
    generate_embedding

2025-01-12 01:52:50.675668: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-12 01:52:50.740690: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-12 01:52:50.742699: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-12 01:52:51.926802: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
dotenv.load_dotenv()

True

In [4]:
# Set the GOOGLE_APPLICATION_CREDENTIALS environment variable
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv('SERVICE_ACCOUNT_PATH')

In [5]:
# Configuration

credentials, project_id = google.auth.default()

PROJECT_ID = project_id
REGION = "asia-southeast2"

# Directory structure
BATCH_ROOT = "batch_root"
EMBEDDINGS_FILE_JSON = 'embeddings.json'
DELETE_DIRECTORY = os.path.join(BATCH_ROOT, "delete")
PDF_FOLDER = "pdf_files"

# GCS Bucket Name and path for embeddings
EMBEDDING_BUCKET_NAME = "sbi-ai-solution-image-embeddings" 
PDF_BUCKET_NAME = f"sample-pdf-{PROJECT_ID}"
IMAGE_BUCKET_NAME = f"{PROJECT_ID}-extracted-images-from-pdf"
GCS_BATCH_ROOT = "batch_root"

INDEX_NAME = "sbi-image-index"
IMAGE_DIRECTORY = "images"
MODEL_URL = "https://tfhub.dev/google/imagenet/resnet_v2_50/feature_vector/5"
NUM_NEIGHBORS = 3

In [6]:
# Initialize Vertex AI SDK
aiplatform.init(project=PROJECT_ID, location=REGION)

In [7]:
# Initialize Google Cloud Storage client
storage_client = storage.Client(project=PROJECT_ID)

In [24]:
# Added Logging Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 1. Prepare Image Dataset

In [25]:
# Ensure required directories exist
os.makedirs(BATCH_ROOT, exist_ok=True)
os.makedirs(DELETE_DIRECTORY, exist_ok=True)
logger.info(f"Created directories: {BATCH_ROOT}, {DELETE_DIRECTORY}")

In [8]:
# Create or get the GCS bucket
embedding_bucket = create_gcs_bucket(EMBEDDING_BUCKET_NAME, REGION, storage_client)
pdf_bucket = create_gcs_bucket(PDF_BUCKET_NAME, REGION, storage_client)
image_bucket = create_gcs_bucket(IMAGE_BUCKET_NAME, REGION, storage_client)

Bucket sbi-ai-solution-image-embeddings already exists.
Bucket sample-pdf-sbi-ai-solution-01 already exists.
Bucket sbi-ai-solution-01-extracted-images-from-pdf already exists.


In [ ]:
# Extract image from pdf

extract_images_from_pdfs_in_gcs(PDF_BUCKET_NAME,IMAGE_BUCKET_NAME)

Processing PDF: Raw Mill Gear Box Report.pdf
  Saved image to gs://sbi-ai-solution-01-extracted-images-from-pdf/images/Raw Mill Gear Box Report_page_2_img_1.jpg
  Saved image to gs://sbi-ai-solution-01-extracted-images-from-pdf/images/Raw Mill Gear Box Report_page_2_img_2.jpg
  Saved image to gs://sbi-ai-solution-01-extracted-images-from-pdf/images/Raw Mill Gear Box Report_page_2_img_3.jpg
  Saved image to gs://sbi-ai-solution-01-extracted-images-from-pdf/images/Raw Mill Gear Box Report_page_2_img_4.jpg
  Saved image to gs://sbi-ai-solution-01-extracted-images-from-pdf/images/Raw Mill Gear Box Report_page_3_img_1.jpg
  Saved image to gs://sbi-ai-solution-01-extracted-images-from-pdf/images/Raw Mill Gear Box Report_page_3_img_2.jpg
  Saved image to gs://sbi-ai-solution-01-extracted-images-from-pdf/images/Raw Mill Gear Box Report_page_4_img_1.jpg
  Saved image to gs://sbi-ai-solution-01-extracted-images-from-pdf/images/Raw Mill Gear Box Report_page_4_img_2.jpg
  Saved image to gs://sbi-a

In [13]:
# Get image paths from the GCS bucket

image_paths = list_gcs_image_paths(image_bucket, IMAGE_DIRECTORY)
if not image_paths:
    print(f"No images found in gs://{image_bucket.name}/{IMAGE_DIRECTORY}. Please put some images in this GCS folder to continue.")
    exit()  # Stop further execution. You have to place images into the folder.

print(f"Found {len(image_paths)} images in the GCS bucket: gs://{image_bucket.name}/{IMAGE_DIRECTORY}")

Found 45 images in the GCS bucket: gs://sbi-ai-solution-01-extracted-images-from-pdf/images


# 2. Generate Image Embeddings

In [14]:
# Load a pre-trained ResNet model from TensorFlow Hub
model = hub.KerasLayer(MODEL_URL)

In [16]:
# Generate Embeddings
embeddings = []

for i, path in enumerate(image_paths):
    embedding = generate_embedding(path, image_bucket, model)
    # Extract file name with extension from GCS path
    file_name_with_extension = os.path.basename(path)

    embedding_record = {
        "id": file_name_with_extension,
        "embedding": embedding.tolist()
    }
    embeddings.append(embedding_record)
    print(f"Generated embedding for image: {os.path.basename(path)}")

Generated embedding for image: Raw Mill Gear Box Report_page_2_img_1.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_2_img_2.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_2_img_3.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_2_img_4.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_3_img_1.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_3_img_2.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_4_img_1.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_4_img_2.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_4_img_3.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_4_img_4.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_5_img_1.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_5_img_2.jpg
Generated embedding for image: Raw Mill Gear Box Report_page_5_img_3.jpg
Generated embedding for image: Raw Mill Gear Box Re

In [19]:
# Save embeddings to JSON file, writing each record on its own line
with open(os.path.join(BATCH_ROOT, EMBEDDINGS_FILE_JSON), "w") as f:
    for record in embeddings:
        json.dump(record, f)
        f.write("\n")
print(f"Generated and saved embeddings to: {os.path.join(BATCH_ROOT, EMBEDDINGS_FILE_JSON)}")

Generated and saved embeddings to: batch_root/embeddings_1.json


In [21]:
# Upload batch_root to GCS
for root, _, files in os.walk(BATCH_ROOT):
    for file in files:
        local_path = os.path.join(root, file)
        relative_path = os.path.relpath(local_path, BATCH_ROOT)
        blob_path = os.path.join(GCS_BATCH_ROOT, relative_path)
        blob = embedding_bucket.blob(blob_path)
        blob.upload_from_filename(local_path)
        print(f"Uploaded {local_path} to gs://{EMBEDDING_BUCKET_NAME}/{blob_path}")

Uploaded batch_root/embeddings_1.json to gs://sbi-ai-solution-image-embeddings/batch_root/embeddings_1.json


In [22]:
# Construct the GCS URI for batch_root
GCS_BATCH_ROOT_URI = f"gs://{EMBEDDING_BUCKET_NAME}/{GCS_BATCH_ROOT}"
print(f"Uploaded batch root to: {GCS_BATCH_ROOT_URI}")

Uploaded batch root to: gs://sbi-ai-solution-image-embeddings/batch_root


# 3. Create Vertex Matching Engine Index

In [ ]:
# Create a vector index for embeddings
index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=INDEX_NAME,
    contents_delta_uri=GCS_BATCH_ROOT_URI,
    dimensions=len(embeddings[0]["embedding"]),
    approximate_neighbors_count=150,
    shard_size="SHARD_SIZE_SMALL"
)

print(f"Created index: {index.display_name}")

In [ ]:
# Create the index endpoint
INDEX_ENDPOINT_NAME = f'{INDEX_NAME}-endpoint'

index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
  display_name=INDEX_ENDPOINT_NAME,
  public_endpoint_enabled=True
)

In [ ]:
#  Deploy the created index.
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
deployed_index_id = f"indexid_sbi_01_{timestamp}"


index_endpoint = index_endpoint.deploy_index(
  index=index,
  deployed_index_id=deployed_index_id,
  machine_type="e2-standard-2",
  min_replica_count=1,
  max_replica_count=1
)

print(f'Deployed endpoint: {index_endpoint.display_name}')

---